# This Notebook is a simple Demonstration of how to use Claude API to build a simple tool call

In [51]:
from dotenv import load_dotenv
load_dotenv()
from anthropic import Anthropic
from datetime import datetime
import json

In [3]:
client = Anthropic()
model = "claude-sonnet-4-0"

# Building Helper Functions

In [5]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages):
    message = client.messages.create(
        model=model,
        max_tokens=1000,
        messages=messages,
    )
    return message.content[0].text

# Execution

In [6]:
messages = []
add_user_message(messages, "Define quantum computing in one sentence")
answer = chat(messages)
print(answer)
add_assistant_message(messages, answer)
add_user_message(messages, "Write a haiku about it")
answer = chat(messages)
print(answer)

Quantum computing is a type of computation that harnesses quantum mechanical phenomena like superposition and entanglement to process information in ways that can potentially solve certain problems exponentially faster than classical computers.
Qubits dance in space,
Superposed between all states—
Computing the impossible.


# Now Lets Build a Demo Chat Module

In [15]:
messages = []
flag = True
while True:
    user_input = input("> ")
    if user_input == "exit":
        break
    print("> ", user_input)
    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print('---')
    print(answer)
    print('---')

>  hi
---
Hello! How are you doing today? Is there anything I can help you with?
---
>  whats 2  2
---
2 + 2 = 4

If you meant a different operation, let me know! For example:
- 2 × 2 = 4
- 2 - 2 = 0
- 2 ÷ 2 = 1
---
>  I ment 2+2 sry
---
No problem at all! Yes, 2 + 2 = 4. 

Is there anything else I can help you with?
---
>  now what is its cube
---
The cube of 4 is 4³ = 4 × 4 × 4 = 64.
---


# Now Lets Ask Claude something it wasn't trained on

In [7]:
messages = []
flag = True
while True:
    user_input = input("> ")
    if user_input == "exit":
        break
    print("> ", user_input)
    add_user_message(messages, user_input)
    answer = chat(messages)
    add_assistant_message(messages, answer)
    print('---')
    print(answer)
    print('---')

>  What is the current weather in Dallas
---
I don't have access to real-time weather data, so I can't tell you the current weather conditions in Dallas. To get up-to-date weather information for Dallas, I'd recommend checking:

- Weather.com or the Weather Channel app
- National Weather Service (weather.gov)
- Local Dallas news websites
- Weather apps on your phone like AccuWeather or Weather Underground

These sources will give you current conditions, temperature, precipitation, and forecasts for Dallas, Texas.
---


# Now Lets solve these type of issues with tool use.

### To do this we must ask claude the question along with some instructions on how to pull this data.

### Lets Try an example of how to set a reminder based on user queries.

### For this lets add 3 tools. 
### 1. Gets the current time. 
### 2. Sets the Target time relative to the current.
### 3. Sets a reminder.

In [19]:
from anthropic.types import ToolParam

def get_current_time(dt_format = "%Y-%m-%d %H:%M:%S"):
    if not dt_format:
        raise ValueError("proper dt_format is required")
    return datetime.now().strftime(dt_format)


get_current_time_schema = ToolParam({
  "name": "get_current_time",
  "description": "Get the current date and time formatted according to the specified format string. Returns a formatted timestamp using Python's strftime directives.",
  "input_schema": {
    "type": "object",
    "properties": {
      "dt_format": {
        "type": "string",
        "description": "Python strftime format string for the output. Common directives: %Y (4-digit year), %m (month), %d (day), %H (24hr hour), %M (minute), %S (second). Example: '%Y-%m-%d %H:%M:%S' returns '2026-03-04 14:30:00'. Must be a non-empty string."
      }
    },
    "required": []
  }
})




# Now lets call the chat module

In [40]:
messages = []
add_user_message(messages, "What is the current time? format it as HH:MM:SS")

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_time_schema]
    )

add_assistant_message(messages, response.content)

print(messages)

[{'role': 'user', 'content': 'What is the current time? format it as HH:MM:SS'}, {'role': 'assistant', 'content': [TextBlock(citations=None, text="I'll get the current time formatted as HH:MM:SS for you.", type='text'), ToolUseBlock(id='toolu_01QVKyBTYpDYCbHCPEz2kDBE', input={'dt_format': '%H:%M:%S'}, name='get_current_time', type='tool_use', caller={'type': 'direct'})]}]


# Now lets retrieve the Input args given by claude

In [41]:
response.content[1].input

{'dt_format': '%H:%M:%S'}

In [43]:
res = get_current_time(**response.content[1].input)

# Now lets give the model the results.

In [44]:
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": response.content[1].id,
        "content": res,
        "is_error": False
    }]
})

In [45]:
response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_time_schema]
    )



In [48]:
response.content[0].text

'The current time is **18:24:32**.'

# Now Lets Bring all this together 

## Let's improve helper functions

In [ ]:
from anthropic.types import Message


def add_user_message(messages, message):
    user_message = {
    "role": "user",
    "content": message.content if isinstance(message, Message) else message
    }
    messages.append(user_message)

def add_assistant_message(messages, message):
    assistant_message = {
    "role": "assistant",
    "content": message.content if isinstance(message, Message) else message
    }
    messages.append(assistant_message)


def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
        "tools": tools
    }
    
    if system:
        params["system"] = system
    
    if tools:
        params["tools"] = tools
    
    message = client.messages.create(**params)
    return message


def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]

    )


def run_tool(tool_name, tool_input):
    if tool_name == "get_current_time":
        return get_current_time(**tool_input)
    else:
        raise ValueError(f"Tool {tool_name} not found")
        


def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []
    
    for tool_request in tool_requests:

        print('here',tool_request.input)
        try:
            tool_output = run_tool(tool_request.name,tool_request.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": json.dumps(tool_output),
                "is_error": False
            })
        except Exception as e:
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": str(e),
        })
            
    return tool_result_blocks

# Now Lets Create a Conversation Cycle 

In [50]:
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_time_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))
        
        if response.stop_reason != "tool_use":
            break
            
        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages

# Now let's ask claude something in 2 tool calls

In [60]:
messages = []
add_user_message(messages, "What is the current date and time? format it as HH:MM:SS.")
run_conversation(messages)



I'll get the current date and time formatted as HH:MM:SS for you.
here {'dt_format': '%H:%M:%S'}
Let me try again with the correct format:
here {'dt_format': '%H:%M:%S'}
I apologize for the error. Let me get the current time for you:
here {'dt_format': '%H:%M:%S'}
I'm experiencing a technical issue with the function call. However, based on your request, you want the current time formatted as HH:MM:SS (hours:minutes:seconds in 24-hour format). The format string "%H:%M:%S" would provide this formatting, showing the time like "14:30:25" for example.

If you could try asking again, I should be able to retrieve the current time for you in that exact format.


[{'role': 'user',
  'content': 'What is the current date and time? format it as HH:MM:SS.'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll get the current date and time formatted as HH:MM:SS for you.", type='text'),
   ToolUseBlock(id='toolu_019ExzS16GJmbFHdnqsWw1VK', input={'dt_format': '%H:%M:%S'}, name='get_current_time', type='tool_use', caller={'type': 'direct'})]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_019ExzS16GJmbFHdnqsWw1VK',
    'content': 'run_tools() takes 1 positional argument but 2 were given'}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Let me try again with the correct format:', type='text'),
   ToolUseBlock(id='toolu_01NmEPWhACSjHrt4Pc9dJre1', input={'dt_format': '%H:%M:%S'}, name='get_current_time', type='tool_use', caller={'type': 'direct'})]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01NmEPWhACSjHrt4Pc9dJre1',
    'content': 'r